# **1.0 Notebook of Initial Exploration and Analysis of the Training Data**
by jack phelan ^_^

## **EDA Report Outline/Checklist**

### Dataset overview
- Dataset source, number of records (rows) and variables (columns), file format, any metadata or documentation available
### Data quality summary
- Summarize the quality and completeness of the raw data
- Identify unusual data points or distributions (outliers)
- Statistical summary of dataset
- Summary statistics for measures (mean, median, std dev) and frequency distributions (counts) for categories 
### Univariate analysis
- Visualizations of each variable (histograms or bar charts).  Can be done with Tableau or Python
- Annotate any significant observations
### Bivariate analysis
- Bivariate visualizations of each variable vs the response variable and any other “interesting” pairs of variables. Can be done with Tableau or Python
Annotate any significant observations
    

## **Data Preparation Report Outline/Checklist**
### Data cleaning steps performed
- Missing data handling
- Outlier handling
- Error correction
- Duplicate removal
### Feature engineering decisions
- Single variable transforms
- New feature definitions
- Categorical encoding decisions
- Normalization/standardization decisions
- Date/time transformations 
### Feature selection decisions
- Variable to remove and rationale
### Dataset partitioning decisions (oversampling/undersampling)
### Final dataset summary
- Number of rows/columns
- Summary of key variables
- Summary of transformations and cleaning steps performed


In [1]:
#imports
import pandas as pd 
import numpy as np


In [2]:
# ingest from file
df = pd.read_csv('../data/raw/healthcare_readmissions_dataset_train.csv')

display(df.head())

,PatientID,Age,Gender,Ethnicity,Hospital ID,Height (m),Smoker,BMI,Weight (kg),Adjusted Weight (kg),Has Diabetes,Has Hypertension,Exercise Frequency,Diet Type,Number of Prior Visits,Medications Prescribed,Length of Stay,Type of Treatment,Readmission within 30 Days
0,1000000,23,Female,African American,Hosp2,1.6,False,25.0,64.0,63.283346,0,0,Regular,High-fat,3.0,3.0,0,NaN,0
1,1000002,56,Female,Hispanic,Hosp3,1.8,True,27.0,87.5,87.678859,0,0,Regular,High-fat,2.0,NaN,2,NaN,0
2,1000003,28,Male,African American,Hosp1,1.8,False,35.0,113.4,113.497844,0,1,NaN,Other,NaN,2.0,5,NaN,0
3,1000004,70,Female,Caucasian,Hosp2,1.8,False,27.7,89.7,89.717694,0,0,NaN,Other,3.0,NaN,0,Major Surgery,0
4,1000005,48,Female,Hispanic,Hosp1,1.9,False,22.4,80.9,80.528927,0,0,Occasional,High-fat,7.0,5.0,7,Major Surgery,1


In [3]:
print(df.shape[0],"Rows", df.shape[1], "Columns")

8038 Rows 19 Columns


In [4]:
display(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 8038 entries, 0 to 8037
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   PatientID                   8038 non-null   int64  
 1   Age                         8038 non-null   int64  
 2   Gender                      8038 non-null   str    
 3   Ethnicity                   8038 non-null   str    
 4   Hospital ID                 8038 non-null   str    
 5   Height (m)                  8038 non-null   float64
 6   Smoker                      8038 non-null   bool   
 7   BMI                         8038 non-null   float64
 8   Weight (kg)                 8038 non-null   float64
 9   Adjusted Weight (kg)        8038 non-null   float64
 10  Has Diabetes                8038 non-null   int64  
 11  Has Hypertension            8038 non-null   int64  
 12  Exercise Frequency          5688 non-null   str    
 13  Diet Type                   8038 non-null   

None

### **Initial Notes on Columns**
- **PatientID**: Unique identifier/tracker; not a feature to use for modeling. 
- **Hospital ID**: Look into it further
- **Height**,**Weight**, and  **BMI** are likely all correlated
- Look into differences between **Weight** and **Adjusted Weight**
- Unify formatting for all columns : )
### **Column Type Breakdown**
- **Measures**: Height, Weight, Adjusted Weight, BMI
- **Booleans**: Gender(?), Smoker, Has Diabetes, Has Hypertension
- **Categorical**: Hospital ID, Excercise Frequency, Diet Type, Numbeer of Prior Visits, Medications Prescribed, Length of Stay
### **Missingness Observations**
- Exercise Frequency: 5688 rows (29.2% missing)
- Number of Prior Visits: 7724 rows (3.9% missing)
- Medications Prescribed: 7381 rows (8.1% missing)
- Type of Treatment: 5552 rows (30.9% missing)

In [5]:
# formatting changes

def format_columns(df : pd.DataFrame) -> pd.DataFrame:
    df_transformed = df.copy()
    
    if 'PatientID' in df_transformed.columns:
        df_transformed = df_transformed.rename(columns={'PatientID': 'patient_id'}) #manual change to align
        
    df_transformed.columns = df_transformed.columns.str.strip().str.lower().str.replace(' ', '_')
    df_transformed.columns = df_transformed.columns.str.replace('(', '').str.replace(')', '')
    
    
    return df_transformed

In [6]:
df_transformed = format_columns(df)

df_transformed

,patient_id,age,gender,ethnicity,hospital_id,height_m,smoker,bmi,weight_kg,adjusted_weight_kg,has_diabetes,has_hypertension,exercise_frequency,diet_type,number_of_prior_visits,medications_prescribed,length_of_stay,type_of_treatment,readmission_within_30_days
0,1000000,23,Female,African American,Hosp2,1.6,False,25.0,64.0,63.283346,0,0,Regular,High-fat,3.0,3.0,0,NaN,0
1,1000002,56,Female,Hispanic,Hosp3,1.8,True,27.0,87.5,87.678859,0,0,Regular,High-fat,2.0,NaN,2,NaN,0
2,1000003,28,Male,African American,Hosp1,1.8,False,35.0,113.4,113.497844,0,1,NaN,Other,NaN,2.0,5,NaN,0
3,1000004,70,Female,Caucasian,Hosp2,1.8,False,27.7,89.7,89.717694,0,0,NaN,Other,3.0,NaN,0,Major Surgery,0
4,1000005,48,Female,Hispanic,Hosp1,1.9,False,22.4,80.9,80.528927,0,0,Occasional,High-fat,7.0,5.0,7,Major Surgery,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8033,1009991,30,Male,Caucasian,Hosp2,1.6,False,19.1,48.9,49.881959,0,0,Regular,Vegetarian,4.0,1.0,3,Major Surgery,0
8034,1009992,45,Female,Caucasian,Hosp3,1.7,False,20.1,58.1,57.904835,1,0,Occasional,Balanced,0.0,2.0,2,Minor Surgery,0
8035,1009994,18,Male,Caucasian,Hosp1,1.7,True,28.5,82.4,82.058378,0,0,Occasional,High-fat,1.0,4.0,8,Minor Surgery,0
8036,1009998,37,Female,Caucasian,Hosp2,1.7,True,25.7,74.3,74.046629,0,0,Regular,Balanced,5.0,4.0,1,Minor Surgery,0


### More Notes
- according to data dictionary excercise_frequency column has 3 distinct values
(including one being 'none'), however dataframe has 2 distincts + missings (with no nones).
most likely case none = missing but will wait for now
- this is also the case with type_of_treament column

- number_of_prior_visits and medications_prescribed both have 0s as well as missings, probably
different from the case mentioned above




In [7]:
#initial data preparation functions

""" 
function for converting booleans that are ints to actual booleans for temporary
visualization/organization purposes
"""

def boolean_type_conversions(df : pd.DataFrame) -> pd.DataFrame:
    df_transformed = df.copy()
    
    if 'has_diabetes' in df_transformed.columns:
        df_transformed['has_diabetes'] = df_transformed['has_diabetes'].astype(bool)
    if 'has_hypertension' in df_transformed.columns:
        df_transformed['has_hypertension'] = df_transformed['has_hypertension'].astype(bool)
    
    return df_transformed

""" 
function to fix the NA encodings for excercise_frequency and type_of_treatment columns
according to the data dictionary 
"""
def fix_na_encodings(df : pd.DataFrame) -> pd.DataFrame:
    df_transformed = df.copy()
    
    if 'exercise_frequency' in df_transformed.columns:
        df_transformed['exercise_frequency'] = df_transformed['exercise_frequency'].replace(np.nan, 'None')
        
    if 'type_of_treatment' in df_transformed.columns:
        df_transformed['type_of_treatment'] = df_transformed['type_of_treatment'].replace(np.nan, 'None')
        
    return df_transformed

""" 
rename target output for clarity : )
"""
def rename_target_column(df : pd.DataFrame) -> pd.DataFrame:
    df_transformed = df.copy()
    
    if 'readmission_within_30_days' in df_transformed.columns:
        df_transformed = df_transformed.rename(columns={'readmission_within_30_days': 'target'})
    return df_transformed

In [8]:
df_transformed = boolean_type_conversions(df_transformed)

df_transformed = fix_na_encodings(df_transformed)

df_transformed = rename_target_column(df_transformed)

display(df_transformed.head())

,patient_id,age,gender,ethnicity,hospital_id,height_m,smoker,bmi,weight_kg,adjusted_weight_kg,has_diabetes,has_hypertension,exercise_frequency,diet_type,number_of_prior_visits,medications_prescribed,length_of_stay,type_of_treatment,target
0,1000000,23,Female,African American,Hosp2,1.6,False,25.0,64.0,63.283346,False,False,Regular,High-fat,3.0,3.0,0,None,0
1,1000002,56,Female,Hispanic,Hosp3,1.8,True,27.0,87.5,87.678859,False,False,Regular,High-fat,2.0,NaN,2,None,0
2,1000003,28,Male,African American,Hosp1,1.8,False,35.0,113.4,113.497844,False,True,None,Other,NaN,2.0,5,None,0
3,1000004,70,Female,Caucasian,Hosp2,1.8,False,27.7,89.7,89.717694,False,False,None,Other,3.0,NaN,0,Major Surgery,0
4,1000005,48,Female,Hispanic,Hosp1,1.9,False,22.4,80.9,80.528927,False,False,Occasional,High-fat,7.0,5.0,7,Major Surgery,1


In [9]:
# review columns after some transformations : )

display(df_transformed.info())

<class 'pandas.DataFrame'>
RangeIndex: 8038 entries, 0 to 8037
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   patient_id              8038 non-null   int64  
 1   age                     8038 non-null   int64  
 2   gender                  8038 non-null   str    
 3   ethnicity               8038 non-null   str    
 4   hospital_id             8038 non-null   str    
 5   height_m                8038 non-null   float64
 6   smoker                  8038 non-null   bool   
 7   bmi                     8038 non-null   float64
 8   weight_kg               8038 non-null   float64
 9   adjusted_weight_kg      8038 non-null   float64
 10  has_diabetes            8038 non-null   bool   
 11  has_hypertension        8038 non-null   bool   
 12  exercise_frequency      8038 non-null   str    
 13  diet_type               8038 non-null   str    
 14  number_of_prior_visits  7724 non-null   float64
 15

None

In [ ]:
df_transformed.to_csv('../data/interim/1.0-initial-cleaned-data.csv', index=False) # logging just in case ^_^

df_transformed

,patient_id,age,gender,ethnicity,hospital_id,height_m,smoker,bmi,weight_kg,adjusted_weight_kg,has_diabetes,has_hypertension,exercise_frequency,diet_type,number_of_prior_visits,medications_prescribed,length_of_stay,type_of_treatment,target
0,1000000,23,Female,African American,Hosp2,1.6,False,25.0,64.0,63.283346,False,False,Regular,High-fat,3.0,3.0,0,None,0
1,1000002,56,Female,Hispanic,Hosp3,1.8,True,27.0,87.5,87.678859,False,False,Regular,High-fat,2.0,NaN,2,None,0
2,1000003,28,Male,African American,Hosp1,1.8,False,35.0,113.4,113.497844,False,True,None,Other,NaN,2.0,5,None,0
3,1000004,70,Female,Caucasian,Hosp2,1.8,False,27.7,89.7,89.717694,False,False,None,Other,3.0,NaN,0,Major Surgery,0
4,1000005,48,Female,Hispanic,Hosp1,1.9,False,22.4,80.9,80.528927,False,False,Occasional,High-fat,7.0,5.0,7,Major Surgery,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8033,1009991,30,Male,Caucasian,Hosp2,1.6,False,19.1,48.9,49.881959,False,False,Regular,Vegetarian,4.0,1.0,3,Major Surgery,0
8034,1009992,45,Female,Caucasian,Hosp3,1.7,False,20.1,58.1,57.904835,True,False,Occasional,Balanced,0.0,2.0,2,Minor Surgery,0
8035,1009994,18,Male,Caucasian,Hosp1,1.7,True,28.5,82.4,82.058378,False,False,Occasional,High-fat,1.0,4.0,8,Minor Surgery,0
8036,1009998,37,Female,Caucasian,Hosp2,1.7,True,25.7,74.3,74.046629,False,False,Regular,Balanced,5.0,4.0,1,Minor Surgery,0
